In [1]:
import pathlib

import yaml
import scipy.signal
import pandas as pd
import numpy as np

import jax.random as jr
from jaxtyping import Array, Float

import torch
from torch.utils.data import DataLoader

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets
# from sklearn.manifold import TSNE

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
data_directory = pathlib.Path.cwd().parent / 'data'
assert data_directory.exists()

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = '4_classes_modulated'
# base_filename = '20_600'
base_filename = '63_225'

signals_file = data_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

# metadata_file = data_directory / f'{base_filename}_labels.parquet'
metadata_file = data_directory / f'{base_filename}_metadata.parquet'
assert metadata_file.exists()

# signals_file, metadata_file

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

trapallada.data - WARNING - dismissed 198 constant signals (that cannot be normalized); 14885 left


The directory containing the actual results

In [6]:
# results_directory = pathlib.Path.cwd()
# results_directory = pathlib.Path.cwd() / 'results'
results_directory = pathlib.Path('/tmp')
assert results_directory.exists()

In [7]:
list(results_directory.glob('*.eqx'))

[PosixPath('/tmp/1.eqx'),
 PosixPath('/tmp/2.eqx'),
 PosixPath('/tmp/3.eqx'),
 PosixPath('/tmp/4.eqx'),
 PosixPath('/tmp/5.eqx'),
 PosixPath('/tmp/real_z_20_4000_epochs.eqx'),
 PosixPath('/tmp/real_z_50_4000_epochs.eqx'),
 PosixPath('/tmp/smaller.eqx')]

*Base* file name for the results (trained model)

In [8]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_50_300_epochs' # <-------------------------------------
# input_files_stem = results_directory / 'new' / 'z_2_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_3_300_epochs' 
# input_files_stem = results_directory / 'new' / 'real_z_50_300_epochs'
# ---
# input_files_stem = results_directory / 'real_z_50_4000_epochs'
# input_files_stem = results_directory / 'real_z_20_4000_epochs'
# input_files_stem = results_directory / 'smaller'
# input_files_stem = results_directory / '1'
# input_files_stem = results_directory / '2' # <---- good
# input_files_stem = results_directory / '3'
input_files_stem = results_directory / '4' # <---- good
# input_files_stem = results_directory / '5' # <---- overfitting


In [9]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [10]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [11]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [12]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# parameters
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 225
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 50 
  kernel_sizes:
  - 64
  - 32
  - 16
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 4000
  train_test_split:
    test_size: 0.2


In [13]:
results_df = pd.read_parquet(results_file)
results_df

training                            validation                \
             loss loglikelihood         kl         loss loglikelihood   
0     1779.534038  -1761.269316  18.264723  1726.402713  -1708.386540   
1     1715.128164  -1697.504506  17.623662  1710.772669  -1693.448845   
2     1707.026495  -1690.194474  16.832020  1705.656228  -1689.214159   
3     1703.112445  -1687.295833  15.816611  1702.438319  -1687.171757   
4     1699.989033  -1685.542243  14.446787  1698.628184  -1684.981628   
...           ...           ...        ...          ...           ...   
3995  1183.284576  -1093.248309  90.036269  1217.035446  -1126.607900   
3996  1183.851048  -1093.697213  90.153833  1216.238362  -1126.488348   
3997  1183.444606  -1093.319200  90.125403  1217.289027  -1127.534871   
3998  1183.216243  -1093.144723  90.071519  1217.171879  -1127.340659   
3999  1183.525640  -1093.462492  90.063147  1217.383444  -1127.355042   

                 
             kl  
0     18.016170  
1     17.323822  
2     16.442061  
3     15.266561  
4     13.646560  
...         ...  
3995  90.427544  
3996  89.750018  
3997  89.754162  
3998  89.831218  
3999  90.028404  

[4000 rows x 6 columns]

# Training curves

In [14]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [15]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [16]:
sum(trapallada.util.get_layers_n_parameters(model).values())

29789

Data loaders for both the training and validation sets

In [17]:
# train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
#     ds,
#     parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
#     parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [18]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

torch.Size([32, 1, 225])

Is every class present?

In [19]:
batch[2].unique()

tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan], dtype=torch.float64)

An element from the above batch is encoded, decoded, and a prediction plotted

In [20]:
# i = 0
# i = 1
i = 31

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

start            2019-11-22 00:14:14.532324
end              2019-11-22 05:57:38.052324
tar file                     0853980901.tar
source      P0853980901PNS001SRCTSR8002.FTZ
Name: 169, dtype: object

alt.Chart(...)

## Javi's paper

In [21]:
javi_df = ds.metadata_df[ds.metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ']
javi_df.tail()

,start,end,tar file,source
5914,2009-06-03 10:45:23.497454,2009-06-03 16:28:47.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5918,2009-06-03 12:02:02.497454,2009-06-03 17:45:26.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5922,2009-06-03 13:18:41.497454,2009-06-03 19:02:05.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5926,2009-06-03 14:35:20.497454,2009-06-03 20:18:44.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5930,2009-06-03 15:19:47.917454,2009-06-03 21:03:11.437454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ


In [22]:
(javi_df['end'] - javi_df['start']).tail()

5914   0 days 05:43:23.520000
5918   0 days 05:43:23.520000
5922   0 days 05:43:23.520000
5926   0 days 05:43:23.520000
5930   0 days 05:43:23.520000
dtype: timedelta64[ns]

In [23]:
n_signals_to_plot = 1
trapallada.plot.signals_array(ds[javi_df.tail(n_signals_to_plot).index][1].numpy().squeeze(axis=1))

alt.Chart(...)

In [24]:
i_javi = javi_df.tail(1).index.item()
plot_prediction(ds, i_javi)

start            2009-06-03 15:19:47.917454
end              2009-06-03 21:03:11.437454
tar file                     0605950101.tar
source      P0605950101PNS003SRCTSR8001.FTZ
Name: 5930, dtype: object

alt.Chart(...)

Was this signal seen during training?

In [25]:
i_javi in train_ds[:][0]
# i_javi in validation_ds[:][0]

True

# Latent space projection

In [26]:
df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
10938,2005-04-21 22:27:05.000873,2005-04-22 04:10:28.520873,0305480101.tar,P0305480101PNS003SRCTSR8003.FTZ,-1.967436,-0.200151,1.051384,-2.342838,1.943317,1.376170,...,-0.768994,0.782881,0.063607,-0.113649,-1.027189,-0.136503,0.094260,-1.023711,-8.906382,14.469031
5219,2012-03-19 15:40:27.798108,2012-03-19 21:23:51.318108,0671850301.tar,P0671850301PNS001SRCTSR8002.FTZ,0.362431,1.995024,-0.601961,-0.756980,0.123983,0.362360,...,0.435354,-0.305954,-1.140537,0.914281,0.775414,-1.266816,-1.566068,0.998813,-8.390342,-21.383303
3745,2018-01-14 10:31:39.038451,2018-01-14 16:15:02.558451,0803240501.tar,P0803240501PNS003SRCTSR8004.FTZ,-0.741782,-2.140576,0.773574,-0.373229,-0.192845,-2.081663,...,-0.608502,0.072382,-0.522322,-0.252771,0.788592,-0.554259,-0.968276,0.113199,0.237202,4.022348
7639,2022-08-18 19:33:53.144691,2022-08-19 01:17:16.664691,0902760801.tar,P0902760801PNS003SRCTSR8004.FTZ,-1.174298,-0.829053,-0.881352,-1.628751,-1.203761,1.228029,...,-1.213486,-0.603757,-2.995143,-1.560818,0.115144,0.769021,1.552802,-0.915765,-8.415833,2.234507
2301,2004-01-13 08:18:56.579036,2004-01-13 14:02:20.099036,0201160401.tar,P0201160401PNS003SRCTSR8005.FTZ,-0.260471,-2.068707,-0.493581,0.073513,-0.285387,-1.139254,...,-0.229589,0.446576,-0.173439,-0.460753,0.863067,0.341988,0.389696,1.057887,-4.768647,15.615514


Projections also for the training set

In [27]:
df_train = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df_train.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
98,2018-01-18 06:39:08.007996,2018-01-18 12:22:31.527996,0810600201.tar,P0810600201PNS003SRCTSR8003.FTZ,-0.350584,-0.399266,1.905028,2.145694,-1.481422,-1.117918,...,0.526157,0.723522,1.684330,-0.433067,-0.298728,0.024587,0.245720,-1.807273,16.956053,-23.516375
14544,2009-06-20 13:09:18.777333,2009-06-20 18:52:42.297333,0604310201.tar,P0604310201PNS003SRCTSR8004.FTZ,-1.279979,1.515019,0.437587,-0.691993,0.850934,-0.345971,...,-0.118443,0.722478,-0.094969,0.435868,1.091144,-0.071815,-0.600633,-1.963516,-8.284412,-44.823929
6279,2009-04-01 06:09:16.616242,2009-04-01 11:52:40.136242,0554750401.tar,P0554750401PNS003SRCTSR8004.FTZ,0.660143,-0.848267,-0.009487,1.184932,0.025676,0.425123,...,0.045862,0.413123,1.801326,0.383098,0.657659,-0.526727,-0.529230,-0.814461,-14.747421,-33.236721
3734,2018-01-14 08:52:00.338451,2018-01-14 14:35:23.858451,0803240501.tar,P0803240501PNS003SRCTSR8001.FTZ,-0.872173,1.246706,-1.302447,0.428784,-0.266145,0.734410,...,-0.124711,-0.853509,0.901721,-0.507229,0.822773,0.935833,-0.828845,-0.586228,35.796303,-8.974092
5164,2017-11-25 11:03:35.937376,2017-11-25 16:46:59.457376,0803952401.tar,P0803952401PNS003SRCTSR8004.FTZ,0.826894,-0.183702,0.035023,0.395261,-0.253100,0.747109,...,-0.272074,1.180239,0.145397,0.968132,-0.126360,-0.523268,-1.572331,-0.411510,-16.597004,9.099198


In [31]:
df_train.loc[i_javi, ['tsne_1', 'tsne_2']]

tsne_1    32.722885
tsne_2     28.69433
Name: 5930, dtype: object

In [29]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='tar file:N',
    fill='tar file:N',
    tooltip=['source:N']
    # tooltip=['tar file:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

Signals that are just noise have no SNR

In [30]:
df['SNR'].isna().sum()

KeyError: 'SNR'

As a *fix*, their SNR is set to the maximum

In [ ]:
df.loc[df['label'] == 'noise', 'SNR'] = df['SNR'].max()

...plotting

In [ ]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

The latent space is projected into a 3D space

In [ ]:
df_3 = trapallada.util.make_projections_df(ds, validation_dl, model.encoder, n_projected_components=3)
df_3.head()

In [ ]:
import plotly.express as px
fig = px.scatter_3d(df_3, x='tsne_1', y='tsne_2', z='tsne_3', color='label', opacity=0.5, width=800, height=800)
fig.show()

# Real data

In [ ]:
# signals_file = data_directory / '0605950101.pt'
signals_file = data_directory / '0605950101_bin_size_3.pt'
assert signals_file.exists()

In [ ]:
real_ds = trapallada.data.SignalsDataset(signals_file, metadata_filename=None, normalize=True)

In [ ]:
real_ds[3:5]

Data loaders for both the training and validation sets

In [ ]:
torch_rng = torch.Generator().manual_seed(parameters['PRNG seeds']['training dataloader'])
real_dl = DataLoader(real_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)

A batch

In [ ]:
batch = next(iter(real_dl))
batch

In [ ]:
signals = batch[1]
signals.shape

Let us drop the last few measurements so that the signals have the appropriate length for the trained model

In [ ]:
signals = signals[..., :10_000]
signals.shape

In [ ]:
n_samples_to_plot = 2000
trapallada.plot.signals_array(signals.squeeze()[:, :n_samples_to_plot])

A signal to *upsample* signals

In [ ]:
def upsample_signals(signals: Float[Array, 'batch 1 time'], target_size: int) -> Float[Array, 'batch 1 time']:
    
    upsampling_factor = np.ceil(target_size / signals.shape[-1])
    
    return scipy.signal.resample_poly(signals, upsampling_factor, 1, axis=-1)[..., :target_size]

In [ ]:
# resampled_signals = upsample_signals(signals, 10_000)
# resampled_signals.shape

In [ ]:
# n_samples_to_plot = 2_000
# trapallada.plot.signals_array(resampled_signals[..., :n_samples_to_plot].squeeze())

In [ ]:
# resampled_signals[0].shape

In [ ]:
i = 12
x_pred, z_mean, z_sd = model(signals[i].numpy(), jr.PRNGKey(42))

# first dimension of `x_pred` is 1; below it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': signals[i].ravel()})

trapallada.plot.df(df.iloc[:n_samples_to_plot])